In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.mixture import GaussianMixture

rng = np.random.default_rng(42)
data = np.concatenate([rng.normal(2, 1.0, 300), rng.normal(8, 1.5, 200)])


def gaussian(x, mu, sigma):
    return np.exp(-0.5 * ((x - mu) / sigma) ** 2) / (sigma * np.sqrt(2 * np.pi))


mu = np.array([1.0, 6.0])
sigma = np.array([1.0, 1.0])
pi = np.array([0.5, 0.5])

prev_ll = -np.inf
for iteration in range(1, 201):
    # E-step
    weighted = np.vstack([pi[k] * gaussian(data, mu[k], sigma[k]) for k in range(2)])
    resp = weighted / weighted.sum(axis=0)

    # M-step
    Nk = resp.sum(axis=1)
    mu = (resp * data).sum(axis=1) / Nk
    sigma = np.sqrt((resp * (data - mu[:, None]) ** 2).sum(axis=1) / Nk)
    pi = Nk / len(data)

    log_likelihood = np.log(weighted.sum(axis=0)).sum()
    if iteration % 10 == 0 or iteration == 1:
        print(f"Iteration {iteration:3d}  log-likelihood = {log_likelihood:.4f}")
    if abs(log_likelihood - prev_ll) < 1e-6:
        print(f"Converged at iteration {iteration}")
        break
    prev_ll = log_likelihood

print("\nEM results")
print("Means             :", np.round(mu, 3))
print("Standard deviations:", np.round(sigma, 3))
print("Mixing weights    :", np.round(pi, 3))

gmm = GaussianMixture(n_components=2, random_state=0).fit(data.reshape(-1, 1))
order = np.argsort(gmm.means_.ravel())
print("\nscikit-learn GaussianMixture")
print("Means             :", np.round(gmm.means_.ravel()[order], 3))
print("Standard deviations:", np.round(np.sqrt(gmm.covariances_.ravel())[order], 3))
print("Mixing weights    :", np.round(gmm.weights_[order], 3))

x = np.linspace(data.min() - 1, data.max() + 1, 500)
plt.hist(data, bins=30, density=True, alpha=0.5, label="Data")
plt.plot(x, sum(pi[k] * gaussian(x, mu[k], sigma[k]) for k in range(2)),
         "r", label="EM fitted mixture")
plt.title("Expectation-Maximization")
plt.legend()
plt.show()